# ResNet-18 — MNIST 7 vs. 9

Toy-Notebook: Ein **ResNet-18** (an 1-Kanal-MNIST 28x28 angepasst) lernt, die Ziffern **7** und **9** zu unterscheiden. Laeuft lokal (CPU / Apple-MPS / CUDA wird automatisch erkannt).

## 1. Imports & Konfiguration

In [ ]:
import ssl
from pathlib import Path

# macOS-Python bringt oft keine CA-Zertifikate mit -> MNIST-Download schlaegt
# mit CERTIFICATE_VERIFY_FAILED fehl. certifi-Bundle nutzen, sonst Pruefung lockern.
try:
    import certifi
    ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
except ImportError:
    ssl._create_default_https_context = ssl._create_unverified_context

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18

import matplotlib.pyplot as plt

# 7 und 9 werden auf die Klassen 0 und 1 abgebildet.
DIGITS = (7, 9)
DATA_DIR = Path("data")
WEIGHTS_PATH = Path("mnist_7v9_resnet18.pth")

EPOCHS = 3
BATCH_SIZE = 128
LR = 1e-3

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("Geraet:", device)

## 2. Daten — nur die Ziffern 7 und 9

Wir laden MNIST, behalten nur die Ziffern 7 und 9 und schreiben die Labels auf **0 (=7)** und **1 (=9)** um.

In [ ]:
def make_loader(train, batch_size):
    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    ds = datasets.MNIST(root=str(DATA_DIR), train=train, download=True, transform=tfm)
    mask = (ds.targets == DIGITS[0]) | (ds.targets == DIGITS[1])
    idx = torch.where(mask)[0]
    ds.targets = (ds.targets == DIGITS[1]).long()  # 7 -> 0, 9 -> 1
    return DataLoader(Subset(ds, idx.tolist()), batch_size=batch_size,
                      shuffle=train, num_workers=2)

train_loader = make_loader(train=True, batch_size=BATCH_SIZE)
test_loader  = make_loader(train=False, batch_size=BATCH_SIZE)
print(f"Train-Bilder: {len(train_loader.dataset)} | Test-Bilder: {len(test_loader.dataset)}")

### Beispielbilder

In [ ]:
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lab in zip(axes, imgs, labels):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title("7" if lab.item() == 0 else "9")
    ax.axis("off")
plt.show()

## 3. Modell — ResNet-18 fuer MNIST

Standard-ResNet-18 aus `torchvision`, angepasst:
- **`conv1`** -> 1 Eingangskanal, 3x3-Stem mit stride 1
- **`maxpool`** entfernt (sonst schrumpfen die 28x28-Bilder zu frueh)
- **`fc`** -> 2 Klassen

In [ ]:
def build_model():
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model

model = build_model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

## 4. Training

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * y.size(0)
    train_loss = running / len(train_loader.dataset)
    acc = evaluate(model, test_loader)
    print(f"Epoche {epoch}/{EPOCHS} | Loss {train_loss:.4f} | Test-Acc {acc:.4f}")

## 5. Gewichte speichern

In [ ]:
torch.save(model.state_dict(), WEIGHTS_PATH)
print("Gewichte gespeichert:", WEIGHTS_PATH.resolve())

## 6. Vorhersagen anschauen

Ein paar Testbilder mit der Modell-Vorhersage (gruen = richtig, rot = falsch).

In [ ]:
imgs, labels = next(iter(test_loader))
model.eval()
with torch.no_grad():
    preds = model(imgs.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lab, pred in zip(axes, imgs, labels, preds):
    name = lambda c: "7" if c == 0 else "9"
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(name(pred.item()), color="green" if pred == lab else "red")
    ax.axis("off")
plt.show()